# Permutation-seed robustness of the contextual persistence law

**Reviewer concern (memo item 2):** every CPF estimate subtracts a *single* shuffled
permutation of the prior context. A single permutation could be an unusually easy or hard
ordering, so the fitted exponent might depend on the chosen seed. This notebook rules that out.

**What it does.** For each (target, context length) it draws **K independent permutations**
and records the target perplexity under *every* one of them (not just their mean). One run
therefore yields two things:

1. an **averaged-shuffle** CPF (shuffled perplexity averaged over K permutations) and its
   refit exponent, versus the original single-permutation exponent; and
2. a **seed-sensitivity** distribution — treating each individual permutation as "the one
   permutation", giving K estimates of every corpus exponent.

**The three diagnostics the reviewer asked for** are printed at the end:
the cross-corpus mean exponent stays near 1; per-corpus exponent spread across seeds is small;
and the corpus ranking / fit quality do not depend on the seed.

**Faithfulness.** Same probe, tokenizer, log-spaced context lengths, 30-token target at the
50% position, and min-length filter as `Corpus_Expansion_LongRange_Llama`. Permutation index
**k = 0 uses seed `SEED + c`, exactly reproducing the original single-permutation baseline**, so
the new run is anchored to the published headline (mean alpha = 1.04).

**Config knobs (next cell):** `MODE='sensitivity'` (subset of docs, tractable) vs `'full'`
(all documents, ~10x compute); `K` permutations; `PROBE` = `'llama'` (headline) or `'mistral'`
(probe replication).

**Output:** `My Drive/LRTIA/Results/corpus_expansion_longrange_permrobust/<probe>/`.

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

import numpy as np, json, math, os, gc, random, time
from pathlib import Path
from scipy import stats
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/LRTIA')
except Exception:
    DRIVE = Path(os.environ.get('LRTIA_DRIVE', './LRTIA'))

# ------------------------------- CONFIG -------------------------------
PROBE = 'llama'          # 'llama' (headline probe) or 'mistral' (replication)
MODEL_MAP = {
    'llama':   'unsloth/Meta-Llama-3.1-8B',
    'mistral': 'mistralai/Mistral-7B-v0.1',
}
MODEL_NAME = MODEL_MAP[PROBE]

K = 20                   # independent permutations per (target, context length)
MODE = 'sensitivity'     # 'sensitivity' = subset of docs; 'full' = all documents (~10x compute)
N_DOCS_SUBSET = 25       # docs per corpus when MODE == 'sensitivity' (ignored when 'full')

CTX_LENGTHS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
MAX_CTX   = max(CTX_LENGTHS)
TARGET_LEN  = 30
TARGET_FRAC = 0.5
MIN_DOC_TOK = MAX_CTX + TARGET_LEN + 50
D_MIN_FIT   = 10.0
SEED = 20260503          # perm 0 -> SEED+c reproduces the original single-permutation baseline

# Document enumeration (model-agnostic doc lists; text is re-tokenized per probe).
TARGETS_PATH      = DRIVE / 'Results/corpus_expansion/targets_llama.jsonl'
TOK_MANIFEST_PATH = DRIVE / 'Results/corpus_expansion/tokenized_manifest_llama.jsonl'

BASE = DRIVE / f'Results/corpus_expansion_longrange_permrobust/{PROBE}'
BASE.mkdir(parents=True, exist_ok=True)

RUN_CORPORA = [
    'gutenberg_fiction_en', 'ted_transcripts_en', 'ted_transcripts_de',
    'literary_ja', 'literary_fi', 'news_en',
    'ted_transcripts_fr', 'ted_transcripts_tr',
    # 'buckeye',              # spontaneous speech; add for the 9th cell if desired
]
# The Russian cell lives in a separate source; add 'ted_transcripts_ru' if present in this Drive.

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
docs_note = N_DOCS_SUBSET if MODE == 'sensitivity' else 'ALL'
print('GPU:', gpu)
print('Probe:', PROBE, '(' + MODEL_NAME + ')')
print('K =', K, '| MODE =', MODE, '| docs/corpus =', docs_note)
print('Output dir:', BASE)

In [ ]:
# Load probe in fp16 (matches the original long-range run; skip 4-bit on A100/H100).
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto',
)
model.eval()
print(MODEL_NAME, 'loaded (fp16)')

In [ ]:
# Perplexity of the fixed 30-token target given a context prefix (identical to the headline run).
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf')
    return math.exp(nll / cnt)

def compute_curves_permset(full_ids, ts, te):
    '''Ordered ppl (len C) and shuffled ppl for K permutations (K x C).
       Permutation 0 uses seed SEED+c, reproducing the original single-shuffle baseline.'''
    tgt = full_ids[ts:te]
    ordered = []
    shuf = [[] for _ in range(K)]
    for c in CTX_LENGTHS:
        pfx = [] if c == 0 else full_ids[ts - c:ts]
        p_ord = ppl_nll(pfx, tgt)
        ordered.append(p_ord)
        if c == 0:
            for k in range(K):
                shuf[k].append(p_ord)            # nothing to shuffle at c = 0
        else:
            for k in range(K):
                rng = random.Random(SEED + c + k * 10007)   # k=0 -> SEED+c (original)
                s = list(pfx); rng.shuffle(s)
                shuf[k].append(ppl_nll(s, tgt))
    return ordered, shuf

print('Pipeline ready. Forward passes per target =',
      len(CTX_LENGTHS), 'ordered +', (len(CTX_LENGTHS) - 1) * K, 'shuffled')

In [ ]:
# Document resolvers (reused verbatim from Corpus_Expansion_LongRange_Llama).
tok_manifest = {}
if TOK_MANIFEST_PATH.exists():
    with open(TOK_MANIFEST_PATH) as f:
        for line in f:
            d = json.loads(line)
            tok_manifest[d['document_id']] = d['file_path']

all_targets = []
if TARGETS_PATH.exists():
    with open(TARGETS_PATH) as f:
        for line in f:
            all_targets.append(json.loads(line))
ce_corpora = {}
for t in all_targets:
    ce_corpora.setdefault(t['corpus_id'], set()).add(t['document_id'])

ML_DATA = DRIVE / 'Data/multilingual_literary'
def literary_docs(lang):
    m = json.loads((ML_DATA / 'manifests' / f'{lang}.json').read_text(encoding='utf-8'))
    for author in m['authors']:
        for t in author['texts']:
            fp = ML_DATA / t['text_path']
            try:
                yield t['text_id'], fp.read_text(encoding='utf-8', errors='replace').strip()
            except Exception:
                continue

BUCKEYE_JSONL = DRIVE / 'Data/buckeye_processed/speaker_concatenated.jsonl'
def buckeye_docs():
    if not BUCKEYE_JSONL.exists():
        return
    with open(BUCKEYE_JSONL) as f:
        for line in f:
            d = json.loads(line)
            txt = d.get('text', '').strip()
            if txt:
                yield d['doc_id'], txt

def fix_ce_path(p):
    return str(p).replace('data/corpus_expansion/clean/',
        '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')

def ce_docs(corpus_id):
    for doc_id in ce_corpora.get(corpus_id, set()):
        fp = tok_manifest.get(doc_id)
        if fp is None:
            continue
        abs_fp = Path(fix_ce_path(fp))
        if not abs_fp.exists():
            abs_fp = Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion') / Path(fp).name
        try:
            yield doc_id, abs_fp.read_text(encoding='utf-8', errors='replace').strip()
        except Exception:
            continue

def get_doc_texts(corpus_id):
    if corpus_id == 'buckeye':
        yield from buckeye_docs()
    elif corpus_id.startswith('literary_'):
        yield from literary_docs(corpus_id.split('_', 1)[1])
    else:
        yield from ce_docs(corpus_id)

for c in RUN_CORPORA:
    print(' ', c + ':', sum(1 for _ in get_doc_texts(c)), 'docs found')

In [ ]:
# Main run loop. Deterministic doc order (sorted by id) so the subset is reproducible.
for corpus_id in RUN_CORPORA:
    cache_path = BASE / f'{corpus_id}.json'
    if cache_path.exists():
        print(corpus_id, ': cached, skipping')
        continue
    docs = sorted(get_doc_texts(corpus_id), key=lambda x: x[0])
    if not docs:
        print(corpus_id, ': no docs')
        continue

    results = []; used = 0; t0 = time.time()
    for doc_id, text in tqdm(docs, desc=corpus_id):
        if MODE == 'sensitivity' and used >= N_DOCS_SUBSET:
            break
        full_ids = tokenizer.encode(text, add_special_tokens=False)
        if len(full_ids) < MIN_DOC_TOK:
            continue
        ts = int(MAX_CTX + TARGET_FRAC * ((len(full_ids) - TARGET_LEN) - MAX_CTX))
        te = ts + TARGET_LEN
        if ts - MAX_CTX < 0 or te > len(full_ids):
            continue
        ordered, shuf = compute_curves_permset(full_ids, ts, te)
        results.append({
            'corpus_id': corpus_id, 'document_id': doc_id,
            'target_id': f'{doc_id}__pos50', 'target_frac': TARGET_FRAC,
            'context_lengths': list(CTX_LENGTHS),
            'ordered_ppl': ordered,
            'shuffled_ppl_perm': shuf,   # K x C
            'K': K, 'seed': SEED,
        })
        used += 1

    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print(' ', corpus_id, ':', len(results), 'targets in',
          round((time.time() - t0) / 60, 1), 'min ->', cache_path.name)

print('Run complete.')

In [ ]:
# ---- Analysis helpers ----
import pandas as pd
from scipy.stats import spearmanr

def marginals(ppl, cs):
    out = []
    for i in range(1, len(cs)):
        if cs[i - 1] == 0:
            continue
        w = cs[i] - cs[i - 1]
        d = math.sqrt(cs[i - 1] * cs[i])
        out.append((d, (ppl[i - 1] - ppl[i]) / w))
    return out

def corpus_Pd(records, shuf_of):
    '''shuf_of(rec) -> shuffled ppl array (len C). Returns aggregated (d_array, P_array).'''
    cs = records[0]['context_lengths']
    acc = {}
    for r in records:
        om = marginals(r['ordered_ppl'], cs)
        sm = marginals(shuf_of(r), cs)
        for i, ((d, orr), (_, shr)) in enumerate(zip(om, sm)):
            acc.setdefault(i, []).append((d, orr, shr))
    ds, Ps = [], []
    for i in sorted(acc):
        rows = acc[i]; n = len(rows)
        ds.append(rows[0][0])
        Ps.append(sum(x[1] for x in rows) / n - sum(x[2] for x in rows) / n)
    return np.array(ds), np.array(Ps)

def fit_alpha(d, P):
    m = (d >= D_MIN_FIT) & (P > 0)
    if m.sum() < 4:
        return None, None
    s, _, r, _, _ = stats.linregress(np.log(d[m]), np.log(P[m]))
    return -s, r ** 2          # alpha = -slope

def mean_shuf(rec):
    return list(np.mean(np.array(rec['shuffled_ppl_perm']), axis=0))

def perm_shuf(rec, k):
    return rec['shuffled_ppl_perm'][k]

records = {}
for corpus_id in RUN_CORPORA:
    fp = BASE / f'{corpus_id}.json'
    if fp.exists():
        recs = json.load(open(fp))
        if recs:
            records[corpus_id] = recs
Krun = records[list(records)[0]][0]['K']
col_avg = f'alpha_avg(K={Krun})'
print('Loaded corpora:', list(records), '| K =', Krun)

In [ ]:
# ---- (1) Averaged-shuffle exponent vs original single-permutation exponent ----
rows = []
for c, recs in records.items():
    d,  P  = corpus_Pd(recs, mean_shuf)
    a_avg, r_avg = fit_alpha(d, P)
    d0, P0 = corpus_Pd(recs, lambda r: perm_shuf(r, 0))   # perm 0 == original seed
    a0, r0 = fit_alpha(d0, P0)
    rows.append({'corpus': c,
                 'alpha_single_seed0': a0, 'r2_single': r0,
                 col_avg: a_avg, 'r2_avg': r_avg,
                 'delta_alpha': (a_avg - a0) if (a_avg and a0) else None})
avg_df = pd.DataFrame(rows)
print(avg_df.round(3).to_string(index=False))
print()
print('Mean alpha  single-seed:', round(avg_df['alpha_single_seed0'].mean(), 3),
      '  averaged:', round(avg_df[col_avg].mean(), 3))

In [ ]:
# ---- (2) Seed-sensitivity: each permutation treated as 'the' permutation ----
seed_rows = []
per_seed_crosscorpus = []   # cross-corpus mean alpha for each individual seed
for k in range(Krun):
    a_by_corpus = {}
    for c, recs in records.items():
        d, P = corpus_Pd(recs, lambda r, kk=k: perm_shuf(r, kk))
        a, _ = fit_alpha(d, P)
        if a is not None:
            a_by_corpus[c] = a
    per_seed_crosscorpus.append(np.mean(list(a_by_corpus.values())))

for c, recs in records.items():
    alphas = []
    for k in range(Krun):
        d, P = corpus_Pd(recs, lambda r, kk=k: perm_shuf(r, kk))
        a, _ = fit_alpha(d, P)
        if a is not None:
            alphas.append(a)
    alphas = np.array(alphas)
    seed_rows.append({'corpus': c, 'alpha_mean': alphas.mean(), 'alpha_sd': alphas.std(),
                      'alpha_min': alphas.min(), 'alpha_max': alphas.max(),
                      'n_seeds': len(alphas)})
seed_df = pd.DataFrame(seed_rows)
per_seed_crosscorpus = np.array(per_seed_crosscorpus)
print(seed_df.round(3).to_string(index=False))
print()
print('Cross-corpus mean alpha across the', Krun, 'seeds:',
      round(per_seed_crosscorpus.mean(), 3), '+/-', round(per_seed_crosscorpus.std(), 3),
      '(range', round(per_seed_crosscorpus.min(), 3), '-', round(per_seed_crosscorpus.max(), 3), ')')

In [ ]:
# ---- (3) Ranking / fit stability + verdict + save ----
avg_alpha = {r['corpus']: r[col_avg] for _, r in avg_df.iterrows()}
ref_order = [c for c, _ in sorted(avg_alpha.items(), key=lambda x: x[1])]

corr_list = []
for k in range(Krun):
    ak = {}
    for c, recs in records.items():
        d, P = corpus_Pd(recs, lambda r, kk=k: perm_shuf(r, kk))
        a, _ = fit_alpha(d, P); ak[c] = a
    common = [c for c in ref_order if ak.get(c) is not None]
    rho, _ = spearmanr([avg_alpha[c] for c in common], [ak[c] for c in common])
    corr_list.append(rho)
corr_list = np.array(corr_list)

max_sd = float(seed_df['alpha_sd'].max())
seed_mean = float(per_seed_crosscorpus.mean())
avg_mean = float(avg_df[col_avg].mean())

print('=' * 64)
print('PERMUTATION-SEED ROBUSTNESS  -  VERDICT')
print('=' * 64)
print('1. Cross-corpus mean alpha stays near 1:', round(seed_mean, 3),
      '(across', Krun, 'seeds; averaged-baseline', round(avg_mean, 3), ')')
print('2. Per-corpus exponent spread across seeds is small: max SD =', round(max_sd, 3))
print('3. Corpus ranking is stable: Spearman(seed vs averaged) min =',
      round(float(corr_list.min()), 3), 'mean =', round(float(corr_list.mean()), 3))

summary = {
    'probe': PROBE, 'model': MODEL_NAME, 'K': int(Krun), 'mode': MODE,
    'n_docs_per_corpus': (N_DOCS_SUBSET if MODE == 'sensitivity' else 'all'),
    'cross_corpus_mean_alpha_seeds': seed_mean,
    'cross_corpus_mean_alpha_averaged': avg_mean,
    'max_per_corpus_alpha_sd': max_sd,
    'ranking_spearman_min': float(corr_list.min()),
    'per_corpus': seed_df.to_dict(orient='records'),
    'averaged_vs_single': avg_df.to_dict(orient='records'),
}
json.dump(summary, open(BASE / 'permutation_robustness_summary.json', 'w'), indent=2)
avg_df.to_csv(BASE / 'averaged_vs_single_exponents.csv', index=False)
seed_df.to_csv(BASE / 'seed_sensitivity_exponents.csv', index=False)
print('Saved summary + CSVs to', BASE)

In [ ]:
# ---- Optional SI figure: per-seed P(d) (grey) vs K-permutation average (blue) ----
import matplotlib.pyplot as plt
ncol = 4; nrow = int(np.ceil(len(records) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow), squeeze=False)
for ax, (c, recs) in zip(axes.flat, records.items()):
    for k in range(Krun):
        d, P = corpus_Pd(recs, lambda r, kk=k: perm_shuf(r, kk))
        m = (d >= D_MIN_FIT) & (P > 0)
        ax.plot(d[m], P[m], color='0.75', lw=0.6, alpha=0.6)
    d, P = corpus_Pd(recs, mean_shuf); m = (d >= D_MIN_FIT) & (P > 0)
    ax.plot(d[m], P[m], 'o-', color='C0', ms=4)
    a, _ = fit_alpha(d, P)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.grid(alpha=0.3)
    ax.set_title(f'{c}\nalpha_avg = {a:.2f}', fontsize=9)
for ax in axes.flat[len(records):]:
    ax.axis('off')
fig.suptitle(f'Per-seed P(d) (grey) vs {Krun}-permutation average (blue)  -  {PROBE}', y=1.02)
fig.tight_layout()
fig.savefig(BASE / 'permutation_robustness_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure to', BASE / 'permutation_robustness_curves.png')